# SAM 3 Teacher Cache Workflow

SAM 3のtext promptを少数画像で診断してから、train全件のteacher cacheを生成します。先に `docs/sam3-teacher.md` に従って専用環境を準備し、このNotebookのkernelに `.venv-sam3` を選択してください。

In [ ]:
import subprocess
import sys
from pathlib import Path


def run(*arguments: str) -> None:
    subprocess.run([sys.executable, "-m", "fm_to_edge_seg", *arguments], check=True)


run("doctor")

`cuda_available: True`とGPU名を確認します。次のパスとpromptを環境に合わせて変更してください。業務画像では `thread` と `yarn` を別々のcacheへ出して比較します。

In [ ]:
MANIFEST = Path("data/deepcrack/manifest.csv")
PROMPT = "crack"
DEBUG_CACHE = Path("teacher_cache/sam3_text_crack_debug")
FULL_CACHE = Path("teacher_cache/sam3_text_crack")

## 1. 3枚だけ生成して目視確認

In [ ]:
run(
    "create-sam3-teacher-cache",
    str(MANIFEST),
    str(DEBUG_CACHE),
    "--prompt",
    PROMPT,
    "--score-threshold",
    "0.5",
    "--max-samples",
    "3",
)

In [ ]:
run(
    "preview-teacher-cache",
    str(MANIFEST),
    str(DEBUG_CACHE),
    "artifacts/sam3_text_debug.png",
    "--limit",
    "3",
)

`artifacts/sam3_text_debug.png`を開き、緑のteacher領域とconfidenceを確認します。promptやthresholdを変更する場合は、別のcache名で再実行してください。

In [ ]:
run(
    "evaluate-teacher-cache",
    str(MANIFEST),
    str(DEBUG_CACHE),
    "artifacts/sam3_text_debug_evaluation",
    "--confidence-threshold",
    "0.5",
)

## 2. train全件のcache生成
少数画像の結果を承認できた場合だけ実行します。

In [ ]:
run(
    "create-sam3-teacher-cache",
    str(MANIFEST),
    str(FULL_CACHE),
    "--prompt",
    PROMPT,
    "--score-threshold",
    "0.5",
)

## 3. E003 Student学習
configのcache pathが `FULL_CACHE` と一致していることを確認します。

In [ ]:
run("train", "configs/experiment/e003_sam3_text_logit_kd.yaml")